# Pickle dependencies of top Hugging Face models

In [41]:
import asyncio
import html
import json
import os
import re
from typing import Optional
import aiohttp

In [2]:
http = aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=100, ttl_dns_cache=300))

## List top models

In [44]:
async def http_get(url: str) -> tuple[any, any]:
    retry_count = 0
    while True:
        async with http.get(url) as response:
            if response.status == 404:
                return (None, None)

            if response.status == 429:
                await asyncio.sleep(3)
                continue

            if response.status >= 500 and retry_count < 10:
                retry_count += 1
                await asyncio.sleep(3)
                continue

            if response.status >= 400:
                raise RuntimeError(f"HTTP error: {response.status}\n{await response.text()}")

            return (await response.read(), response.headers)

async def get_page(url: str) -> tuple[any, Optional[str]]:
    data, headers = await http_get(url)
    return (
        json.loads(data.decode('utf-8')) if data is not None else None,
        headers['Link'].split(';')[0].strip('<>') if headers is not None and 'Link' in headers else None)
        
async def get_top_models(model_count: int = 1_000) -> list[dict]:
    models = []
    page_url = 'https://huggingface.co/api/models?sort=downloads&direction=-1&limit=100'
    while len(models) < model_count:
        page_models, page_url = await get_page(page_url)
        models.extend(page_models)
        if page_url is None or len(models) >= model_count:
            return models

In [45]:
models = await get_top_models()

In [46]:
os.makedirs('data', exist_ok=True)

with open('data/models.jsonl', 'w') as f:
    for model in models:
        f.write(json.dumps(model) + '\n')

## Get model Pickle globals

In [52]:
with open('data/models.jsonl', 'r') as f:
    model_ids = [json.loads(line)['id'] for line in f]

assert len(model_ids) == 1_000, f"Unexpected array length: {len(model_ids)}"

In [63]:
async def get_model_info(model_id: str) -> dict:
    # Get the model info by downloading the model HTML page and extracting the data-props attribute
    # (there is probably a better way to do this via some REST API but I have not found it)
    data, _ = await http_get(f"https://huggingface.co/{model_id}/tree/main")
    model_info_line = next(line for line in data.decode('utf-8').split("\n") if 'data-target="ViewerIndexTreeList"' in line)
    return json.loads(html.unescape(re.search(r'data-props="([^"]+)"', model_info_line).group(1)))

async def get_model_security_info(model_id: str) -> list[dict]:
    model_info = await get_model_info(model_id)
    model_id = model_info["context"]["repo"]["name"];
    return [{
        "model_id": model_id,
        "path": item["path"],
        "safe": item["security"]["safe"],
        "virusFound": item["security"]["avScan"]["virusFound"] if item["security"]["avScan"] is not None else None,
        "virusNames": item["security"]["avScan"]["virusNames"] if item["security"]["avScan"] is not None else None,
        "pickleSafetyLevel": item["security"]["pickleImportScan"]["highestSafetyLevel"] if item["security"]["pickleImportScan"] is not None else None,
        "pickleImports": item["security"]["pickleImportScan"]["imports"] if item["security"]["pickleImportScan"] is not None else None,
    } for item in model_info["entries"] if item["type"] == "file" and item.get("security") is not None]

async def get_model_pickle_files(model_id: str) -> list[str]:
    model_info = await get_model_security_info(model_id)
    return [file for file in model_info if file.get('pickleImports', None) is not None]

In [64]:
# Test on well-known issue
await get_model_pickle_files('ykilcher/totally-harmless-model')

[{'model_id': 'ykilcher/totally-harmless-model',
  'path': 'pytorch_model.bin',
  'safe': False,
  'virusFound': False,
  'virusNames': None,
  'pickleSafetyLevel': 'dangerous',
  'pickleImports': [{'module': 'torch._utils',
    'name': '_rebuild_tensor_v2',
    'safety': 'innocuous'},
   {'module': '__builtin__', 'name': 'eval', 'safety': 'dangerous'},
   {'module': 'collections', 'name': 'OrderedDict', 'safety': 'innocuous'},
   {'module': 'torch', 'name': 'FloatStorage', 'safety': 'innocuous'}]}]

In [76]:
pickle_import_stats = {}
for i, model_id in enumerate(model_ids):
    try:
        print(f"Querying model {i+1}/{len(model_ids)} {model_id}                                                                ", end="\r")
        pickle_files = await get_model_pickle_files(model_id)
        for pickle_file in pickle_files:
            for pickle_import in pickle_file['pickleImports']:
                pickle_import_id = pickle_import['module'] + '|' + pickle_import['name']
                stats = pickle_import_stats.get(pickle_import_id, {'count': 0, 'safety': pickle_import['safety']})
                stats['count'] += 1
                pickle_import_stats[pickle_import_id] = stats
    except Exception as e:
        print(f"Error while processing {model_id}: {e}")

print(f"Found {len(pickle_import_stats)} unique pickle imports")

Error while processing michellejieli/NSFW_text_classifier: coroutine raised StopIteration                                                                       
Found 98 unique pickle importsnki-NLP/opus-mt-tr-en                                                                                                                    


In [77]:
with open('data/pickle_import_stats.json', 'w') as f:
    json.dump(pickle_import_stats, f)

In [88]:
for row in sorted(pickle_import_stats.items(), key=lambda x: x[1]['count'], reverse=True):
    if row[1]['count'] > 1:
        print(row)

('torch._utils|_rebuild_tensor_v2', {'count': 818, 'safety': 'innocuous'})
('collections|OrderedDict', {'count': 818, 'safety': 'innocuous'})
('torch|FloatStorage', {'count': 634, 'safety': 'innocuous'})
('torch|LongStorage', {'count': 244, 'safety': 'innocuous'})
('torch|HalfStorage', {'count': 119, 'safety': 'innocuous'})
('torch|BFloat16Storage', {'count': 72, 'safety': 'innocuous'})
('torch|device', {'count': 71, 'safety': 'suspicious'})
('transformers.trainer_utils|IntervalStrategy', {'count': 67, 'safety': 'suspicious'})
('transformers.trainer_utils|SchedulerType', {'count': 67, 'safety': 'suspicious'})
('transformers.training_args|TrainingArguments', {'count': 59, 'safety': 'suspicious'})
('transformers.trainer_utils|HubStrategy', {'count': 55, 'safety': 'suspicious'})
('transformers.training_args|OptimizerNames', {'count': 49, 'safety': 'suspicious'})
('torch|ByteStorage', {'count': 29, 'safety': 'innocuous'})
('torch|IntStorage', {'count': 28, 'safety': 'innocuous'})
('acceler